In [1]:
## Reranking Hybrid Search Strategies

In [27]:
from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [28]:
## Load Text File
loader = TextLoader("langchain_sample.txt")
raw_docs = loader.load()

## Split text into document chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

In [29]:
chunks

[Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses. LangChain makes it easy to implement RAG using vector databases like FAISS, Chroma, and Pinecone.\nBM25 is a traditional 

In [30]:
## User Query
query  = "How can I use langchain to build an application with memory and tools?"

In [31]:
### FAISS and HuggingFace model embeddings

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs = {"k": 4})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9363.91it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
## OpenAI Embedding
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()
vectorstore_openai = FAISS.from_documents(chunks, embeddings)
retriever_openai = vectorstore_openai.as_retriever(search_kwargs  ={"k" : 4})

In [33]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000247340D2300>, search_kwargs={'k': 4})

In [34]:
retriever_openai

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000247339C8B00>, search_kwargs={'k': 4})

In [35]:
## Prompt and use the LLM
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
# gemma2-9b-it was retired by Groq; see https://console.groq.com/docs/deprecations
llm = init_chat_model("llama-3.1-8b-instant", model_provider="groq")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000247339CACC0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000247339CB2F0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [36]:
# Prompt Template
prompt = PromptTemplate.from_template("""
You are a helpful assistant. Your task is to rank the following documents from most to least relevant.

User question: {question}

Documents:
{documents}

Instructions:
- Think about the relevance of each document to the user's questions.
- Return a list of document indices in ranked order, starting from the most relevant.

Output format: Comma-separated document indices (e.g., 2,1,3,0,...)
""")

In [37]:
retriever_docs = retriever.invoke(query)
retriever_docs

[Document(id='7971080b-9a84-4a01-b47f-684683558750', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='0c72f977-ecb7-422c-815f-c9100aa03138', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='08750aeb-cff1-42c3-b346-192d3593ec8b', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging F

In [38]:
chain  =prompt | llm | StrOutputParser()
chain

PromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, template="\nYou are a helpful assistant. Your task is to rank the following documents from most to least relevant.\n\nUser question: {question}\n\nDocuments:\n{documents}\n\nInstructions:\n- Think about the relevance of each document to the user's questions.\n- Return a list of document indices in ranked order, starting from the most relevant.\n\nOutput format: Comma-separated document indices (e.g., 2,1,3,0,...)\n")
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000247339CACC0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000247339CB2F0>, model_name='llama-3.1-8b-instant', model

In [39]:
doc_lines = [f"{i+1}. {doc.page_content}" for i, doc in enumerate(retriever_docs)]
formatted_docs = "\n".join(doc_lines)

In [40]:
doc_lines

['1. LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.',
 '2. LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.',
 '3. LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.',
 '4. FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spaces. It supports both flat and comp

In [41]:
formatted_docs

'1. LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.\n2. LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.\n3. LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.\n4. FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spaces. It supports both flat and compressed ind

In [42]:
response = chain.invoke({"question":query, "documents": formatted_docs})
response

"Based on the user's question, I will rank the documents from most to least relevant. \n\nThe user is asking about using LangChain to build an application with memory and tools, which implies they are looking for information on the framework's capabilities and features related to tool integration, memory, and application development.\n\nDocument 2 directly mentions LangChain's capabilities, including memory, tool integration, prompt management, chains, and agents, making it highly relevant to the user's question.\n\nDocument 1 is also highly relevant as it specifically mentions LangChain's support for tool integration and memory, which are key components of the user's requested application.\n\nDocument 4 is not relevant to the user's question as it discusses FAISS, a separate library, which is not mentioned in the user's query.\n\nDocument 3 is somewhat relevant as it mentions LangChain's integration with third-party services, but it does not specifically address the user's question ab

In [45]:
## Step 5: Parse and rerank
indices = [int(x.strip()) -1 for x in response.split(",") if x.strip().isdigit()]
reranked_docs = [retriever_docs[i] for i in indices if 0 <= i < len(retriever_docs)]
reranked_docs

[Document(id='7971080b-9a84-4a01-b47f-684683558750', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='08750aeb-cff1-42c3-b346-192d3593ec8b', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.')]

In [44]:
# Step 6. Show results
print("\n Final Reranked Results")
for i,doc in enumerate(reranked_docs, 1):
    print(f"\nRank {i}: \n{doc.page_content}")
    


 Final Reranked Results

Rank 1: 
LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.
Memory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.

Rank 2: 
LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.
